In [1]:
import pandas as pd 
import numpy as np
import os
import json
import re

In [2]:
"""
1. Read in the data
- Cazy

""" 

# 列出ProCAST.xlsx的所有sheet
print(pd.ExcelFile("ProCAST.xlsx").sheet_names)

['TCDB', 'KOG', 'DFVF', 'P450', 'CAZy', 'NR', 'SwissProt', 'TIGRFAM', 'SFLD', 'PHOBIUS', 'SIGNALP_EUK', 'SUPERFAMILY', 'PANTHER', 'GENE3D', 'HAMAP', 'PROSITE_PROFILES', 'PROSITE_PATTERNS', 'COILS', 'SMART', 'CDD', 'PRINTS', 'PFAM', 'MOBIDB_LITE', 'PIRSF', 'TMHMM', 'PRODOM', 'GO', 'PATHWAY', 'SeqLen']


In [5]:
# 读取ProCAST.xlsx
df = pd.read_excel("ProCAST.xlsx", sheet_name="CAZy")
df

,query_id,subject_id,identity,alignment length,mismatches,gap,q.start,q.end,s.start,s.end,e-value,bit score
0,XP_003658326.1,QCD77328.1 GH36: Glycoside Hydrolases (GHs),52.5,120,53,1,48,167,747,862,1.700000e-26,123.6
1,XP_003658327.1,AEO53082.1 GH2: Glycoside Hydrolases (GHs),100.0,855,0,0,1,855,1,855,0.000000e+00,1791.5
2,XP_003658329.1,VUW51410.1 GT4: GlycosylTransferases (GTs),50.7,207,89,5,75,280,493,687,3.700000e-46,189.5
3,XP_003658332.1,QCD77328.1 GH36: Glycoside Hydrolases (GHs),43.1,153,82,2,29,181,748,895,2.600000e-28,129.0
4,XP_003658346.1,AAF79504.1 GH36: Glycoside Hydrolases (GHs) AA...,43.4,145,81,1,580,723,995,1139,2.700000e-21,107.5
...,...,...,...,...,...,...,...,...,...,...,...,...
765,XP_003667369.1,AEO62124.1 GH27: Glycoside Hydrolases (GHs),100.0,416,0,0,1,416,1,416,2.300000e-256,887.5
766,XP_003667383.1,AEO62138.1 GH47: Glycoside Hydrolases (GHs),100.0,632,0,0,1,632,1,632,0.000000e+00,1255.0
767,XP_003667394.1,BBN14963.1 CBM20: Carbohydrate-Binding Modules...,45.7,398,201,7,13,403,394,783,1.700000e-95,353.2
768,XP_003667406.1,AEO62161.1 GH5_7: Glycoside Hydrolases (GHs),100.0,410,0,0,1,410,1,410,3.600000e-249,863.6


In [7]:
# 读取ProCAST.xlsx
df = pd.read_excel("ProCAST.xlsx", sheet_name="GO")
df.to_csv("ProCAST_GO.csv", index=False)

,id,GO
0,XP_003658317.1,protein binding|MOLECULAR_FUNCTION|GO:0005515
1,XP_003658319.1,acid phosphatase activity|MOLECULAR_FUNCTION|G...
2,XP_003658319.1,acid phosphatase activity|MOLECULAR_FUNCTION|G...
3,XP_003658319.1,hydrolase activity|MOLECULAR_FUNCTION|GO:0016787
4,XP_003658322.1,nucleic acid binding|MOLECULAR_FUNCTION|GO:000...
...,...,...
8234,XP_003667406.1,"hydrolase activity, hydrolyzing O-glycosyl com..."
8235,XP_003667408.1,transmembrane transporter activity|MOLECULAR_F...
8236,XP_003667408.1,transmembrane transport|BIOLOGICAL_PROCESS|GO:...
8237,XP_003667408.1,transmembrane transport|BIOLOGICAL_PROCESS|GO:...


In [17]:
# 读取CSV文件
df = pd.read_csv('ProCAST_GO.csv')

# 拆分GO列中的多个条目
df['GO'] = df['GO'].str.split(';')

# 扩展DataFrame以包含多个条目
df = df.explode('GO')

# 拆分GO列中的信息
df[['TERM', 'ONTOLOGY', 'GO_id']] = df['GO'].str.split('|', expand=True)

# 选择所需的列
df = df[['id', 'GO_id', 'TERM', 'ONTOLOGY']]

df.rename(columns={'GO_id':'GO','ONTOLOGY':'GO_type', 'TERM':'ONTOLOGY'}, inplace=True)

df

,id,GO,ONTOLOGY,GO_type
0,XP_003658317.1,GO:0005515,protein binding,MOLECULAR_FUNCTION
1,XP_003658319.1,GO:0003993,acid phosphatase activity,MOLECULAR_FUNCTION
1,XP_003658319.1,GO:0046872,metal ion binding,MOLECULAR_FUNCTION
2,XP_003658319.1,GO:0003993,acid phosphatase activity,MOLECULAR_FUNCTION
3,XP_003658319.1,GO:0016787,hydrolase activity,MOLECULAR_FUNCTION
...,...,...,...,...
8237,XP_003667408.1,GO:0016020,membrane,CELLULAR_COMPONENT
8237,XP_003667408.1,GO:0022857,transmembrane transporter activity,MOLECULAR_FUNCTION
8238,XP_003667408.1,GO:0055085,transmembrane transport,BIOLOGICAL_PROCESS
8238,XP_003667408.1,GO:0016021,integral component of membrane,CELLULAR_COMPONENT


In [19]:
go_df = df.groupby('id').agg({
    'GO': lambda x: ','.join(x),
    'ONTOLOGY': lambda x: ','.join(x),
    'GO_type': lambda x: ','.join(x)
}).reset_index()

go_df

,id,GO,ONTOLOGY,GO_type
0,XP_003658317.1,GO:0005515,protein binding,MOLECULAR_FUNCTION
1,XP_003658319.1,"GO:0003993,GO:0046872,GO:0003993,GO:0016787","acid phosphatase activity,metal ion binding,ac...","MOLECULAR_FUNCTION,MOLECULAR_FUNCTION,MOLECULA..."
2,XP_003658322.1,GO:0003676,nucleic acid binding,MOLECULAR_FUNCTION
3,XP_003658323.1,"GO:0051287,GO:0055114,GO:0016616,GO:0051287,GO...","NAD binding,oxidation-reduction process,oxidor...","MOLECULAR_FUNCTION,BIOLOGICAL_PROCESS,MOLECULA..."
4,XP_003658325.1,GO:0016491,oxidoreductase activity,MOLECULAR_FUNCTION
...,...,...,...,...
4886,XP_003667403.1,"GO:0016491,GO:0008270,GO:0055114,GO:0016491,GO...","oxidoreductase activity,zinc ion binding,oxida...","MOLECULAR_FUNCTION,MOLECULAR_FUNCTION,BIOLOGIC..."
4887,XP_003667404.1,GO:0005515,protein binding,MOLECULAR_FUNCTION
4888,XP_003667405.1,"GO:0055085,GO:0022857","transmembrane transport,transmembrane transpor...","BIOLOGICAL_PROCESS,MOLECULAR_FUNCTION"
4889,XP_003667406.1,"GO:0004553,GO:0005975","hydrolase activity, hydrolyzing O-glycosyl com...","MOLECULAR_FUNCTION,BIOLOGICAL_PROCESS"


In [23]:
# 将go_df的GO_type列中的MOLECULAR_FUNCTION改为MF
go_df['GO_type'] = go_df['GO_type'].str.replace('MOLECULAR_FUNCTION', 'MF')

# 将go_df的GO_type列中的BIOLOGICAL_PROCESS改为BP
go_df['GO_type'] = go_df['GO_type'].str.replace('BIOLOGICAL_PROCESS', 'BP')

# 将go_df的GO_type列中的CELLULAR_COMPONENT改为CC
go_df['GO_type'] = go_df['GO_type'].str.replace('CELLULAR_COMPONENT', 'CC')

go_df.to_csv('ProCAST_GO.csv', index=False)